# Fine-tuning Qwen2.5-0.5B-Instruct per tool-calling (LoRA, fp16)

Obiettivo: correggere i due problemi emersi testando il modello base con l'harness locale:
1. **confabulazione** (il modello a volte risponde inventando l'esito invece di chiamare un tool),
2. **disciplina del formato** poco affidabile sotto few-shot / su richieste ambigue.

Dataset: `glaiveai/glaive-function-calling-v2`. Le funzioni disponibili variano per ogni esempio del dataset (meteo, calcolatrice, ecc.) — **non sono i nostri tool** (`run_shell`, `list_dir`, `read_file`, `final_answer`). Va bene così: l'obiettivo del fine-tuning e' insegnare il *comportamento* generale (JSON pulito, un tool alla volta, aspettare il risultato, non inventare, usare una risposta finale quando non serve un tool), che si trasferisce a qualsiasi set di tool definito nel system prompt a runtime — esattamente come facciamo nel nostro harness.

Runtime consigliato: **GPU (T4 va bene, A100 se disponibile su Colab Pro)**.

Nota: niente quantizzazione a 4 bit (QLoRA). A 0.5B parametri il modello in fp16 occupa meno di 1.5GB, ben dentro i 15GB della T4 — la quantizzazione qui serve solo a rallentare (overhead di dequantizzazione ad ogni layer, senza reale bisogno di risparmiare VRAM). Usiamo **fp16** invece di bf16 perche' la T4 (architettura Turing) ha tensor core nativi per fp16 ma non per bf16.

In [ ]:
!pip install -q -U transformers datasets peft accelerate trl torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/qwen-toolcalling'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Caricamento e parsing del dataset

Il campo `chat` del dataset e' testo con turni delimitati da `USER:`, `ASSISTANT:`, `FUNCTION RESPONSE:` e marcatori `<|endoftext|>`. Le chiamate a funzione sono nel formato `<functioncall> {"name": "...", "arguments": '{...}'}`. Le convertiamo nel nostro schema `{"tool": ..., "args": {...}}`, e le risposte testuali finali in `{"tool": "final_answer", "args": {"text": ...}}`.

In [ ]:
import re, json, ast, random
from datasets import load_dataset

raw = load_dataset("glaiveai/glaive-function-calling-v2", split="train")
print(len(raw))
print(raw[0]["system"][:500])
print("---")
print(raw[0]["chat"][:800])

In [ ]:
TURN_RE = re.compile(
    r'(USER|ASSISTANT|FUNCTION RESPONSE):\s*(.*?)(?=(?:USER:|ASSISTANT:|FUNCTION RESPONSE:|$))',
    re.DOTALL,
)

def split_turns(chat_text):
    turns = []
    for role, content in TURN_RE.findall(chat_text):
        content = content.replace("<|endoftext|>", "").strip()
        if content:
            turns.append((role, content))
    return turns

def parse_functioncall(text):
    """'<functioncall> {"name": "x", "arguments": '{...}'}' -> (name, args_dict)"""
    body = text.split("<functioncall>", 1)[1].strip()
    obj = ast.literal_eval(body)  # tollera l'uso misto di ' e " nel dataset
    name = obj["name"]
    args = obj["arguments"]
    if isinstance(args, str):
        args = json.loads(args)
    return name, args

def build_system_prompt_from_functions(system_text):
    """Estrae le definizioni di funzione dal campo 'system' del dataset e le
    riscrive nello stesso stile di prompt usato dall'harness locale."""
    body = system_text.split("-", 1)[-1].strip() if "-" in system_text else system_text
    funcs = re.findall(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', body)
    parsed = []
    for f in funcs:
        try:
            data = json.loads(f)
            if isinstance(data, dict):
                parsed.append(data)
        except json.JSONDecodeError:
            continue
    # alcuni "properties" annidati vengono estratti come falsi tool dalla regex
    # sopra (gestisce solo 2 livelli di graffe); se capita, "name" puo' essere
    # un dict (lo schema del parametro) invece di una stringa
    tools_desc = {}
    for i, p in enumerate(parsed):
        name = p.get("name")
        if isinstance(name, dict) or not name:
            name = f"tool_{i}"
        tools_desc[str(name)] = p.get("description", "")
    return (
        "Sei un agente che risolve richieste dell'utente chiamando dei tool.\n\n"
        f"Hai accesso a questi tool:\n{json.dumps(tools_desc, ensure_ascii=False, indent=2)}\n\n"
        "REGOLE FERREE:\n"
        "1. Rispondi SEMPRE E SOLO con un oggetto JSON valido, su una riga, senza testo prima o dopo.\n"
        '2. Formato: {"tool": "<nome_tool>", "args": {...}}\n'
        '3. Per rispondere senza chiamare un tool, usa "final_answer" con il testo in "text".\n'
        "4. Non inventare mai l'esito di un tool: se ti serve un dato, chiamalo, non supporlo.\n"
        "5. Un tool alla volta.\n"
    )

def row_to_messages(row):
    turns = split_turns(row["chat"])
    if not turns:
        return None
    try:
        system_content = build_system_prompt_from_functions(row["system"])
    except Exception:
        return None
    messages = [{"role": "system", "content": system_content}]
    last_tool_name = None
    for role, content in turns:
        if role == "USER":
            messages.append({"role": "user", "content": content})
        elif role == "ASSISTANT":
            if "<functioncall>" in content:
                try:
                    name, args = parse_functioncall(content)
                except Exception:
                    return None
                last_tool_name = name
                payload = json.dumps({"tool": name, "args": args}, ensure_ascii=False)
            else:
                payload = json.dumps({"tool": "final_answer", "args": {"text": content}}, ensure_ascii=False)
            messages.append({"role": "assistant", "content": payload})
        elif role == "FUNCTION RESPONSE":
            messages.append({
                "role": "user",
                "content": f"Risultato del tool '{last_tool_name}':\n{content}",
            })
    # servono almeno un turno utente e un turno assistant per essere un esempio utile
    if sum(m["role"] == "assistant" for m in messages) == 0:
        return None
    return messages

print(row_to_messages(raw[0]))

In [ ]:
N_EXAMPLES = 4000  # sottoinsieme: sufficiente per un 0.5B, tiene i tempi di training bassi su Colab

random.seed(0)
indices = list(range(len(raw)))
random.shuffle(indices)

examples = []
for i in indices:
    if len(examples) >= N_EXAMPLES:
        break
    msgs = row_to_messages(raw[i])
    if msgs:
        examples.append(msgs)

print(f"Esempi validi raccolti: {len(examples)}")
examples[0]

## 2. Tokenizzazione con il chat template di Qwen2.5

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in examples]
ds = Dataset.from_dict({"text": texts})
ds = ds.train_test_split(test_size=0.03, seed=0)
print(ds)
print(ds["train"][0]["text"][:1000])

## 3. LoRA setup e training (fp16, niente quantizzazione)

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="/content/qwen-toolcalling-run",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="epoch",
    fp16=True,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
)
trainer.train()

## 4. Merge dell'adapter LoRA nel modello base e salvataggio su Drive

In [ ]:
adapter_dir = f"{OUTPUT_DIR}/adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

In [ ]:
merged = trainer.model.merge_and_unload()

merged_dir = f"{OUTPUT_DIR}/merged"
merged.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
print("Salvato in", merged_dir)

## 5. Conversione in GGUF (da fare in locale, non su Colab)

Il modello mergiato e' in `MyDrive/qwen-toolcalling/merged` (safetensors). Scaricalo sul PC (o sincronizza la cartella Drive), poi in locale:

```bash
git clone https://github.com/ggerganov/llama.cpp ~/llama.cpp
cd ~/llama.cpp
python3 -m venv venv && source venv/bin/activate
pip install -r requirements.txt

python convert_hf_to_gguf.py /path/to/merged --outfile ~/models/qwen2.5-0.5b-toolcalling-f16.gguf --outtype f16

# quantizzazione Q8_0, coerente con la scelta fatta per il modello base
./llama-quantize ~/models/qwen2.5-0.5b-toolcalling-f16.gguf ~/models/qwen2.5-0.5b-toolcalling-q8_0.gguf Q8_0
```

Poi basta aggiornare `MODEL_PATH` in `agent.py` per puntare al nuovo file, e rilanciare `test_reliability.py` per confrontare i risultati col modello base.